# CSC-321: Data Mining and Machine Learning
# Jordin Palmeri
## Assignment 7: K-Nearest Neighbor

### Part 1: Implementation

For this assignment, I'm going to let you break down the implementation as you see fit. You're going to implement KNN, an example of lazy learning. In brief, this means:

- calculating euclidean distance between the feature values of a single test instance, and the feature values of a single training instance
- making a prediction requires iterating through *all* the training instances, calculating the distances and storing each distance in a list, along with the corresponding class value (probably as some sort of tuple)
- sorting the list, smallest distances first
- selecting the *k* nearest neighbors (where k should be a parameter)
- making a prediction by choosing the class that appears the most in the k nearest neighbors

To calculate euclidean distance between an instance x1 and an instance x2, we need to iterate through the input features of the two instances (for i features) and for each take the difference of (x1[i]) - (x2[i]), squaring that difference, and summing over all features. At the end, take the square root of the total. In other words:

$$distance=\sqrt{\sum_{i=1}^n (x1_{i} - x2_{i})^2}$$


I would strongly suggest you follow the implementation outline of previous algorithms in terms of the functions you use, but I'm leaving it up to you.

Below is the same contrived dataset you've used before. If your code works, you should be able to take an instance of this data, and compare it to all the others (including itself, where the distance SHOULD be 0). Note then for k=1, and using the same data for training and testing, we should always get perfect results.

You should be able to select the k-nearest neighbors, and make a prediction based on the most frequently occuring class in those k neighbors.

Make sure you create a knn function that takes a training set (X_train, y_train), a test set (X_test) and a value for k, that returns a list of predictions - one prediction for each instance in the test set.

Run the algorithm over the sample dataset, using k=3. Print the predicted and the actual side by side.

In [ ]:
import math

# Contrived data set

dataset = [[3.393533211,2.331273381,0],
    [3.110073483,1.781539638,0],
    [1.343808831,3.368360954,0],
    [3.582294042,4.67917911,0],
    [2.280362439,2.866990263,0],
    [7.423436942,4.696522875,1],
    [5.745051997,3.533989803,1],
    [9.172168622,2.511101045,1],
    [7.792783481,3.424088941,1],
    [7.939820817,0.791637231,1]]

def euclidean_distance(instance1, instance2):
    distance = 0
    for i in range(len(instance1) - 1):
        distance += (instance1[i] - instance2[i]) ** 2
    return math.sqrt(distance)


def get_neighbors(X_train, y_train, test_instance, k):
    distances = []
    for i in range(len(X_train)):
        dist = euclidean_distance(test_instance, X_train[i])
        label = y_train[i]
        distances.append((dist, label))

    distances.sort()

    neighbors = [label for (_, label) in distances[:k]]
    return neighbors


def predict_classification(neighbors):
    return max(set(neighbors), key=neighbors.count)


def knn(X_train, y_train, X_test, k):
    predictions = []
    for test_instance in X_test:
        neighbors = get_neighbors(X_train, y_train, test_instance, k)
        prediction = predict_classification(neighbors)
        predictions.append(prediction)
    return predictions

In [ ]:
X_values = [row[:-1] for row in dataset]
y_values = [row[-1] for row in dataset]

predictions = knn(X_values, y_values, X_values, 3)

print("Predicted | Actual")
for pred, actual in zip(predictions, y_values):
    print(f"{pred:^9} | {actual}")

Predicted | Actual
    0     | 0
    0     | 0
    0     | 0
    0     | 0
    0     | 0
    1     | 1
    1     | 1
    1     | 1
    1     | 1
    1     | 1


### Part 2: Working with real data

Apply the KNN algorithm above to the abalone data set. You can find more about it here: http://archive.ics.uci.edu/ml/datasets/Abalone


I want you to use YOUR KNN code. However, I've started the process, because I want to show you another part of scikit learn. I've loaded in the data, and shown the head of the data. Pay attention to the sex column.


In [ ]:
import pandas as pd

labels = ['sex','length','diameter','height','whole_weight','shucked_weight',
          'viscera_weight','shell_weight','rings']

abalone_data = pd.read_csv('https://raw.githubusercontent.com/nixwebb/CSV_Data/master/abalone.csv',names=labels)
abalone_data.head()


,sex,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
0,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


That first column is nominal data, not numeric. We first have to change it into numbers - either replacing each value with an integer (label encoding) or creating new columns to be a binary representation of each possible value (one-hot encoding).

For label encoding, I use the relevant method in scikit learn - although I could first make the column [categorical in pandas](https://pandas.pydata.org/docs/user_guide/categorical.html), and then use the [cat.codes method](https://pandas.pydata.org/docs/reference/api/pandas.Series.cat.codes.html).

For one-hot encoding, if my data is in a pandas dataframe, I prefer the pandas method [get_dummies](https://pandas.pydata.org/docs/reference/api/pandas.get_dummies.html), but I have to remember to DROP the first column (there's an argument - drop_first that I need to set to True).

I'll do one-hot encoding here. I'll use the method in pandas. I'm going to do that in stages, to show you what happens, but you don't need to print after ever step like this, once you're convinced it works.

The get_dummies method in pandas creates my new columns based on feature values. There are three possible feature values for the sex feature (M,F,I - and you should know what these mean), so I create three columns.

Notice that in pandas I can refer to a column using a built in attribute. The abalone_data dataframe has a column labeled 'sex', so I can use abalone_data.sex to access that column. I'm creating column headings from the feature values, and adding the prefix 'sex' to each value.

In [ ]:
abalone_sex = pd.get_dummies(abalone_data.sex, prefix='sex',drop_first=True)
abalone_sex.head()

,sex_I,sex_M
0,False,True
1,False,True
2,False,False
3,False,True
4,True,False


Then I need to add these columns back into my overall dataframe. I'm using the pandas method concat to do that.

In [ ]:
abalone_ohe = pd.concat([abalone_sex,abalone_data],axis=1)
abalone_ohe.head()

,sex_I,sex_M,sex,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
0,False,True,M,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,False,True,M,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,False,False,F,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,False,True,M,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,True,False,I,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


And finally, now I've encoded the sex using one-hot encoding, I'm going to drop the sex column from the dataframe. However, just to show you how the scikit learn label encoder works, execute the code in the first code cell below, and check out what happens to the sex column values. Then run the second cell to drop the sex column from the data.

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
abalone_ohe['sex'] = le.fit_transform(abalone_ohe.sex.values)
abalone_ohe.head()


,sex_I,sex_M,sex,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
0,False,True,2,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,False,True,2,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,False,False,0,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,False,True,2,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,True,False,1,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


In [ ]:
abalone_ohe = abalone_ohe.drop('sex',axis=1)
abalone_ohe.head()

,sex_I,sex_M,length,diameter,height,whole_weight,shucked_weight,viscera_weight,shell_weight,rings
0,False,True,0.455,0.365,0.095,0.5140,0.2245,0.1010,0.150,15
1,False,True,0.350,0.265,0.090,0.2255,0.0995,0.0485,0.070,7
2,False,False,0.530,0.420,0.135,0.6770,0.2565,0.1415,0.210,9
3,False,True,0.440,0.365,0.125,0.5160,0.2155,0.1140,0.155,10
4,True,False,0.330,0.255,0.080,0.2050,0.0895,0.0395,0.055,7


The y value for this data, the thing we're predicting, is the category represented by the number of rings.

Using this dataset, extract X and y data, normalize the X values, and run an UNSTRATIFIED 10-fold cross-validation. Also run a classification baseline. Report on classification accuracy, and write up some results. I want to know how well we did, AND what you think of this as an overall approach.

NOTE: This will be SLOW. If you're not sure your code is working, I recommend starting with a 3 fold cross-validation, and use k=1.

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
from sklearn.dummy import DummyClassifier


X_values_abalone = abalone_ohe.drop('rings', axis=1).values
y_values_abalone = abalone_ohe['rings'].values

rows, cols = X_values_abalone.shape
print(f"This is the abalone data set. It has {rows} instances and {cols} input features.\n")


scaler = MinMaxScaler()

kf = KFold(n_splits=10, shuffle=True, random_state=42)
knn_scores = []
zr_scores = []
k = 3

for train_index, test_index in kf.split(X_values_abalone, y_values_abalone):
    X_train, X_test = X_values_abalone[train_index], X_values_abalone[test_index]
    y_train, y_test = y_values_abalone[train_index], y_values_abalone[test_index]

    scaler.fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --- KNN ---
    preds = knn(X_train_scaled, y_train, X_test_scaled, k)
    accuracy = np.mean(np.array(preds) == y_test)
    knn_scores.append(accuracy)

    # --- ZeroR ---
    dummy = DummyClassifier(strategy='most_frequent')
    dummy.fit(X_train_scaled, y_train)
    dummy_pred = dummy.predict(X_test_scaled)
    dummy_acc = np.mean(dummy_pred == y_test)
    zr_scores.append(dummy_acc)

knn_scores = np.array(knn_scores)
zr_scores = np.array(zr_scores)

knn_avg, knn_min, knn_max, knn_std = knn_scores.mean()*100, knn_scores.min()*100, knn_scores.max()*100, knn_scores.std()*100
zr_avg, zr_min, zr_max, zr_std = zr_scores.mean()*100, zr_scores.min()*100, zr_scores.max()*100, zr_scores.std()*100

print(f"KNN (k={k}): Avg = {knn_avg:.2f}%, Min = {knn_min:.2f}%, Max = {knn_max:.2f}%, Std = {knn_std:.2f}%")
print(f"ZeroR: Avg = {zr_avg:.2f}%, Min = {zr_min:.2f}%, Max = {zr_max:.2f}%, Std = {zr_std:.2f}%")


This is the abalone data set. It has 4177 instances and 9 input features.

KNN (k=3): Avg = 20.71%, Min = 18.71%, Max = 24.16%, Std = 1.66%
ZeroR: Avg = 16.50%, Min = 13.16%, Max = 18.42%, Std = 1.32%


10-Fold Cross-Validation Results (Unstratified)

 - KNN (k=3): Avg = 20.71%, Min = 18.71%, Max = 24.16%, Std = 1.66%
 - ZeroR: Avg = 16.50%, Min = 13.16%, Max = 18.42%, Std = 1.32%

Interpretation:

 - The dataset is challenging because the target variable (rings) has many distinct values and continuous features with overlapping distributions.

 - KNN achieves slightly better performance than the baseline (ZeroR), indicating it is capturing some of the structure in the data.

 - The relatively low KNN accuracy (~21%) is expected when predicting exact counts, not categories. Binning the target into age groups could improve interpretability and accuracy.

 - Standard deviation across folds is small, indicating consistent performance across the splits.


### Part 3: KNN regression

We can also run KNN as a regression algorithm. In this case, instead of predicting the most common class in the k nearest neighbors, we can assign a predicted value that is the **mean** of the values of the k neighbors.

Make this addition to your algorithm (presumably by simply implementing a new predict function below, and then calling this new predict function from your knn algorithm, because you divided your code up sensibly in Part 1), and run the abalone data as a regression problem. To do this, use the same number of folds and the same k value as before. Also run a regression baseline and report evaluation values for both. Give me some explanation of the results, both standalone and in comparison to the classification results above. Which makes more sense for this data?


In [ ]:
from sklearn.metrics import mean_squared_error


def predict_regression(X_train, y_train, test_instance, k):
    neighbors = get_neighbors(X_train, y_train, test_instance, k)
    prediction = sum(neighbors) / len(neighbors)
    return prediction

def knn_regression(X_train, y_train, X_test, k):
    predictions = []
    for test_instance in X_test:
        prediction = predict_regression(X_train, y_train, test_instance, k)
        predictions.append(prediction)
    return predictions


kf = KFold(n_splits=10, shuffle=True, random_state=42)
rmse_scores = []
baseline_rmse = []

for train_index, test_index in kf.split(X_values_abalone, y_values_abalone):
    X_train, X_test = X_values_abalone[train_index], X_values_abalone[test_index]
    y_train, y_test = y_values_abalone[train_index], y_values_abalone[test_index]

    scaler.fit(X_train)
    X_train_scaled = scaler.transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # --- KNN Regression ---
    preds = knn_regression(X_train_scaled, y_train, X_test_scaled, k)
    rmse_scores.append(np.sqrt(mean_squared_error(y_test, preds)))

    # --- ZeroR ---
    dummy = DummyRegressor(strategy='mean')
    dummy.fit(X_train_scaled, y_train)
    baseline_pred = dummy.predict(X_test_scaled)
    baseline_rmse.append(np.sqrt(mean_squared_error(y_test, baseline_pred)))

print(f"KNN Regression (k={k}): RMSE = {np.mean(rmse_scores):.3f}")
print(f"Baseline Regression: RMSE = {np.mean(baseline_rmse):.3f}")


KNN Regression (k=3): RMSE = 2.415
Baseline Regression: RMSE = 3.217


KNN Regression Results (k=3)

 - KNN Regression RMSE: 2.415

 - Baseline Regression RMSE: 3.217

Interpretation:

 - KNN regression achieves a lower RMSE than the baseline, indicating it can make more accurate predictions than simply predicting the mean.

 - Compared to classification (accuracy ~20%), regression is a better fit for this dataset because the target variable, number of rings, is numeric rather than categorical.



## Part 4: Introduction to scikit-learn

One of the most popular open-source python machine learning libraries is scikit-learn. You can find out more in general at: https://scikit-learn.org/stable/index.html


As we go through this class I'll introduce you to some of the functionality. Below I want you to use a KNN Regressor.

I also I want you to explore the cross_val_score function. Previously you used the StratifiedKFold function. You then had to fit the model, then use the predict method to apply the model, then collect the scores, find the mean and the min and the max. cross_val_score does most of that for you. I showed you it in class, but in case you didn't recall that, or have a good picture, this is your opportunity to learn about it.

cross_val_score takes a model (the classifier, or regressor) you want to use, X and y data, a value for cv (the default is 5 for a 5-fold cross validation), and a scoring metric. It does all the fitting, prediction and collection of scores for you.

What is returned is an array of the cross-validation scores. Because it's an array, you can apply the .mean(), .min(), .max() and .std() methods to generate scores as you have before.

If we want to cross-validate a regression algorithm, then we need to change the scoring. By default, the evaluation metric is accuracy. Also by convention, all scoring mechanisms use the idea that LARGER is BETTER (which makes sense with accuracy, but means that scikit needs to do something special for regression. Rather than provide RMSE, it provide NEG_RMSE, so that the convention is preserved. For this experiment, use the parameter scoring='neg_root_mean_squared_error'.

To turn this neg_mean_squared_error into a meaningful score for us, we'll need to take the absolute value (to reverse the sign). Be aware though that the min will be the max, and the max the min. Hopefully you'll see what I mean.

The links to the relevant documentation pages are:
- [KNN Regressor](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html#sklearn.neighbors.KNeighborsRegressor)
- [cross_val_score](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.cross_val_score.html#sklearn.model_selection.cross_val_score)

I'll load the relevant models from scikit-learn, but it's up to you to train and test them, and report the scores appropriately, including comparison to baselines and write up. Your scores should be the broadly the same as your code, above.


In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score
from sklearn.dummy import DummyRegressor


k = 3

# --- KNN Regressor ---
knn_reg = KNeighborsRegressor(n_neighbors=k)

knn_scores = cross_val_score(knn_reg, X_values_abalone, y_values_abalone,
                             cv=10, scoring='neg_root_mean_squared_error')
knn_scores = np.abs(knn_scores)

print("KNN Regression (cross_val_score):")
print(f"RMSE Avg = {knn_scores.mean():.3f}, Min = {knn_scores.min():.3f}, Max = {knn_scores.max():.3f}, Std = {knn_scores.std():.3f}")


# --- Baseline Regressor (mean prediction) ---
baseline_reg = DummyRegressor(strategy='mean')
baseline_scores = cross_val_score(baseline_reg, X_values_abalone, y_values_abalone,
                                  cv=10, scoring='neg_root_mean_squared_error')
baseline_scores = np.abs(baseline_scores)

print("\nBaseline Regression (cross_val_score):")
print(f"RMSE Avg = {baseline_scores.mean():.3f}, Min = {baseline_scores.min():.3f}, Max = {baseline_scores.max():.3f}, Std = {baseline_scores.std():.3f}")


KNN Regression (cross_val_score):
RMSE Avg = 2.310, Min = 1.602, Max = 3.500, Std = 0.715

Baseline Regression (cross_val_score):
RMSE Avg = 3.138, Min = 2.011, Max = 4.598, Std = 0.938


KNN Regression (cross_val_score):
 - RMSE Avg = 2.310, Min = 1.602, Max = 3.500, Std = 0.715

Baseline Regression (cross_val_score):
 - RMSE Avg = 3.138, Min = 2.011, Max = 4.598, Std = 0.938

Interpretation:

 - KNN regression outperforms the baseline mean predictor, with a lower average RMSE (2.310 vs 3.138).

 - The variability across folds (Std = 0.715) is slightly smaller than the baseline (Std = 0.938), indicating somewhat more consistent predictions.

 - The RMSE min/max range shows that in some folds, KNN is very accurate (RMSE = 1.602), while in others it struggles more (RMSE = 3.500).

 - Compared to classification, regression is a more natural fit for predicting the number of abalone rings, since this is a continuous numeric target rather than a discrete class.